# Módulo 04 · Aula 03 — Orientação a Objetos

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Toda função do Atlas recebe `vendas`, `config` e `conexao`. São 14 funções passando os mesmos 3 argumentos. E quando preciso de um estado a mais, mudo a assinatura de todas."*

Orientação a objetos resolve isso agrupando **dados** e **comportamento** que andam juntos.

> ⚠️ **Um aviso antes de começar.** OOP não é "melhor que" funções. É uma ferramenta com casos de uso específicos. Python não é Java: você **não** precisa envolver tudo em classes. Uma função continua sendo a resposta certa na maioria das vezes.
>
> Esta aula ensina a reconhecer os casos em que a classe é a escolha certa — e a evitar a arquitetura inflada que assola tanto código "orientado a objetos".

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | Classes, instâncias, `self` | Agrupar estado e comportamento |
| 2 | Atributos de classe vs de instância | O erro do mutável compartilhado |
| 3 | Encapsulamento e `property` | Proteger invariantes |
| 4 | Métodos de classe e estáticos | Construtores alternativos |
| 5 | Herança | Reuso e especialização |
| 6 | **Composição** | Por que quase sempre é melhor |
| 7 | Métodos dunder | Objetos que se comportam como nativos |
| 8 | Classes abstratas e protocolos | Contratos |
| 9 | Quando **não** usar classes | O mais importante |

## 1. Classe e instância

- **Classe** = a planta da casa
- **Instância** = a casa construída

```python
class NomeDaClasse:
    def __init__(self, parametros):
        self.atributo = valor

    def metodo(self):
        return self.atributo
```

**`self` é a instância atual.** Ele é passado automaticamente quando você chama `objeto.metodo()` — Python traduz isso para `Classe.metodo(objeto)`.

In [ ]:
class Produto:
    """Um produto do catálogo da Aurora."""

    def __init__(self, sku, nome, preco, custo, estoque=0):
        # __init__ NÃO é o construtor — é o INICIALIZADOR.
        # O objeto já existe quando ele roda; ele só preenche os atributos.
        self.sku = sku
        self.nome = nome
        self.preco = preco
        self.custo = custo
        self.estoque = estoque

    def margem(self):
        """Margem em reais."""
        return round(self.preco - self.custo, 2)

    def margem_percentual(self):
        """Margem sobre o preço."""
        return 0.0 if self.preco == 0 else round((self.preco - self.custo) / self.preco, 4)

    def valor_em_estoque(self):
        return round(self.preco * self.estoque, 2)


notebook = Produto("NB-DELL-15", "Notebook Dell Inspiron 15", 2599.90, 2120.00, 14)

print(notebook.nome)
print(f"Margem: R$ {notebook.margem():,.2f} ({notebook.margem_percentual():.1%})")
print(f"Em estoque: R$ {notebook.valor_em_estoque():,.2f}")

In [ ]:
# self é explícito — estas duas chamadas são idênticas
print(notebook.margem())
print(Produto.margem(notebook))    # o que Python faz por baixo

# Cada instância tem seu próprio estado
mouse = Produto("PE-LOG-M170", "Mouse Logitech M170", 89.90, 52.00, 240)
print(f"\n{notebook.sku}: estoque {notebook.estoque}")
print(f"{mouse.sku}: estoque {mouse.estoque}")
print("Estado separado?", notebook.__dict__ is not mouse.__dict__)

## 2. Atributos de classe vs de instância

| | Atributo de **classe** | Atributo de **instância** |
|---|------------------------|---------------------------|
| Onde é definido | No corpo da classe | Dentro de `__init__` (com `self.`) |
| Pertence a | **Todas** as instâncias | Uma instância só |
| Uso típico | Constantes, contadores, configuração | Dados do objeto |

In [ ]:
class Pedido:
    # Atributos de CLASSE — compartilhados
    STATUS_VALIDOS = {"pago", "pendente", "cancelado"}
    TAXA_PADRAO = 0.18
    total_criados = 0

    def __init__(self, cliente, valor, status="pendente"):
        if status not in Pedido.STATUS_VALIDOS:
            raise ValueError(f"status inválido: {status}")
        # Atributos de INSTÂNCIA
        self.cliente = cliente
        self.valor = valor
        self.status = status

        Pedido.total_criados += 1          # altera o da classe
        self.numero = Pedido.total_criados


p1 = Pedido("Maria", 2599.90, "pago")
p2 = Pedido("João", 899.00)
p3 = Pedido("Ana", 1199.00, "pago")

print(f"Pedidos criados: {Pedido.total_criados}")
print(f"Números: {p1.numero}, {p2.numero}, {p3.numero}")
print(f"Taxa (acessível pela instância): {p1.TAXA_PADRAO}")

### 🔴 A armadilha do atributo de classe mutável

Se um atributo de classe for **mutável** (lista, dict, set), **todas** as instâncias compartilham o mesmo objeto. É o irmão gêmeo da armadilha do argumento padrão mutável, da aula 01_04.

In [ ]:
class CarrinhoErrado:
    itens = []            # 🔴 compartilhado por TODAS as instâncias!

    def adicionar(self, item):
        self.itens.append(item)


a = CarrinhoErrado()
b = CarrinhoErrado()
a.adicionar("Notebook")
b.adicionar("Mouse")

print("carrinho a:", a.itens)
print("carrinho b:", b.itens, "  ← 😱 tem o item do outro")
print("Mesmo objeto?", a.itens is b.itens)

In [ ]:
class CarrinhoCerto:
    def __init__(self):
        self.itens = []       # ✅ lista NOVA por instância

    def adicionar(self, item):
        self.itens.append(item)


a = CarrinhoCerto()
b = CarrinhoCerto()
a.adicionar("Notebook")
b.adicionar("Mouse")

print("carrinho a:", a.itens)
print("carrinho b:", b.itens)
print("Mesmo objeto?", a.itens is b.itens)

> 🧭 **Regra:** atributo de classe só para valores **imutáveis** (constantes, tuplas, frozensets). Tudo que é estado do objeto vai em `__init__`.

## 3. Encapsulamento e `property`

Python **não tem** atributos privados de verdade. O que existe é convenção:

| Convenção | Significa |
|-----------|-----------|
| `atributo` | Público — use à vontade |
| `_atributo` | "Interno, não mexa" — só convenção, nada impede |
| `__atributo` | *Name mangling*: vira `_Classe__atributo` |

> 💭 **"Somos todos adultos consentindo aqui"** é o lema da comunidade Python. Em vez de bloquear o acesso, sinalizamos a intenção. Se alguém acessar `_interno`, sabe que está por conta e risco.

### `@property` — atributo calculado

Permite que um **método** seja acessado como se fosse um **atributo**. Isso é ouro: você pode começar com um atributo simples e depois transformá-lo em cálculo, **sem quebrar quem já usa a classe**.

In [ ]:
class ProdutoComProperty:
    def __init__(self, sku, nome, preco, custo, estoque=0):
        self.sku = sku
        self.nome = nome
        self._preco = preco          # _ sinaliza "acesse pela property"
        self.custo = custo
        self.estoque = estoque

    @property
    def preco(self):
        """Getter: acessado como atributo."""
        return self._preco

    @preco.setter
    def preco(self, novo):
        """Setter: valida antes de atribuir."""
        if novo < 0:
            raise ValueError(f"preço não pode ser negativo: {novo}")
        if novo < self.custo:
            raise ValueError(f"preço {novo} abaixo do custo {self.custo}")
        self._preco = novo

    @property
    def margem(self):
        """Somente leitura — calculado a partir de outros atributos."""
        return round(self._preco - self.custo, 2)

    @property
    def situacao(self):
        if self.estoque == 0:
            return "esgotado"
        return "critico" if self.estoque < 10 else "normal"


p = ProdutoComProperty("NB-01", "Notebook", 2599.90, 2120.00, 5)

print(f"preço   : {p.preco}      ← sem parênteses!")
print(f"margem  : {p.margem}")
print(f"situação: {p.situacao}")

p.preco = 2799.90                 # passa pelo setter
print(f"\nnovo preço: {p.preco} | nova margem: {p.margem}")

In [ ]:
for novo_preco in [-100, 1000]:
    try:
        p.preco = novo_preco
    except ValueError as erro:
        print(f"❌ preco = {novo_preco}: {erro}")

# Property sem setter é somente leitura
try:
    p.margem = 999
except AttributeError as erro:
    print(f"❌ margem: {erro}")

> 💡 **Não crie getters e setters por reflexo.** Em Java é convenção envolver todo atributo. Em Python, isso é ruído.
>
> **Comece com o atributo público.** Se um dia precisar de validação ou cálculo, transforme em `property` — e nenhum código que usa a classe precisa mudar. Essa é a razão de a `property` existir.

## 4. Métodos de classe e estáticos

| Decorador | 1º parâmetro | Acessa | Uso típico |
|-----------|--------------|--------|------------|
| (nenhum) | `self` | A instância | Comportamento normal |
| `@classmethod` | `cls` | A classe | **Construtores alternativos** |
| `@staticmethod` | — | Nada | Função relacionada, sem estado |

In [ ]:
class Venda:
    def __init__(self, produto, quantidade, preco_unitario, data):
        self.produto = produto
        self.quantidade = quantidade
        self.preco_unitario = preco_unitario
        self.data = data

    @property
    def total(self):
        return round(self.quantidade * self.preco_unitario, 2)

    # ── Construtores alternativos ──
    @classmethod
    def de_csv(cls, linha):
        """Cria uma Venda a partir de um dict do csv.DictReader."""
        return cls(
            produto=linha["produto"].strip(),
            quantidade=int(linha["quantidade"]),
            preco_unitario=float(linha["preco"]),
            data=linha["data"],
        )

    @classmethod
    def de_texto(cls, texto, separador=";"):
        """Cria a partir de 'produto;qtd;preco;data'."""
        produto, qtd, preco, data = texto.split(separador)
        return cls(produto, int(qtd), float(preco), data)

    # ── Função utilitária relacionada, sem estado ──
    @staticmethod
    def validar_data(texto):
        """Não usa self nem cls — poderia ser função solta, mas
        fica aqui por pertencer conceitualmente à classe."""
        partes = texto.split("-")
        return len(partes) == 3 and len(partes[0]) == 4

    def __repr__(self):
        return f"Venda({self.produto!r}, {self.quantidade}, {self.preco_unitario})"


v1 = Venda("Notebook", 2, 2599.90, "2026-07-01")
v2 = Venda.de_csv({"produto": "Mouse", "quantidade": "10", "preco": "89.90", "data": "2026-07-02"})
v3 = Venda.de_texto("Monitor;3;1199.00;2026-07-03")

for v in (v1, v2, v3):
    print(f"{v}  → total R$ {v.total:,.2f}")

print("\nvalidar_data('2026-07-01'):", Venda.validar_data("2026-07-01"))
print("validar_data('01/07/2026'):", Venda.validar_data("01/07/2026"))

> 💡 **Por que `cls(...)` e não `Venda(...)` dentro do classmethod?** Porque se alguém herdar de `Venda`, o `cls` será a subclasse — e o construtor alternativo continuará funcionando corretamente. Usar o nome fixo quebraria a herança.

## 5. Herança

```python
class Filha(Mae):
    ...
```

A subclasse **herda** todos os atributos e métodos, e pode **sobrescrevê-los**.

`super()` acessa a implementação da classe mãe.

In [ ]:
class Relatorio:
    """Classe base: define o esqueleto comum."""

    extensao = "txt"

    def __init__(self, titulo, dados):
        self.titulo = titulo
        self.dados = dados

    def cabecalho(self):
        return f"{'=' * 50}\n{self.titulo.center(50)}\n{'=' * 50}"

    def corpo(self):
        raise NotImplementedError("cada relatório define o seu corpo")

    def rodape(self):
        return f"{'─' * 50}\n{len(self.dados)} registros"

    def gerar(self):
        """Template method: define a ORDEM, delega o conteúdo."""
        return "\n".join([self.cabecalho(), self.corpo(), self.rodape()])


class RelatorioTexto(Relatorio):
    extensao = "txt"

    def corpo(self):
        linhas = []
        for d in self.dados:
            linhas.append(f"{d['cidade']:<20}{d['valor']:>15,.2f}")
        return "\n".join(linhas)


class RelatorioCSV(Relatorio):
    extensao = "csv"

    def cabecalho(self):
        return "cidade;valor"          # sobrescreve completamente

    def corpo(self):
        return "\n".join(f"{d['cidade']};{d['valor']:.2f}" for d in self.dados)

    def rodape(self):
        return ""                       # CSV não tem rodapé


class RelatorioMarkdown(Relatorio):
    extensao = "md"

    def cabecalho(self):
        # super() reaproveita a lógica da mãe e acrescenta
        return f"# {self.titulo}\n\n| Cidade | Valor |\n|---|---:|"

    def corpo(self):
        return "\n".join(f"| {d['cidade']} | {d['valor']:,.2f} |" for d in self.dados)

    def rodape(self):
        return f"\n_{len(self.dados)} registros_"


dados = [
    {"cidade": "Campinas", "valor": 182450.30},
    {"cidade": "São Paulo", "valor": 298100.00},
    {"cidade": "Sorocaba", "valor": 44200.00},
]

for classe in [RelatorioTexto, RelatorioCSV, RelatorioMarkdown]:
    r = classe("Faturamento por praça", dados)
    print(f"\n{'▼' * 3} {classe.__name__} (.{r.extensao}) {'▼' * 3}")
    print(r.gerar())

> 💡 **O padrão acima tem nome: *Template Method*.** A classe mãe define a **ordem** das etapas (`gerar`), e as filhas preenchem **o conteúdo** de cada uma. É um dos poucos casos em que herança é claramente a ferramenta certa.

In [ ]:
# super() em __init__: estendendo em vez de substituir
class RelatorioComFiltro(RelatorioTexto):
    def __init__(self, titulo, dados, valor_minimo=0):
        dados_filtrados = [d for d in dados if d["valor"] >= valor_minimo]
        super().__init__(titulo, dados_filtrados)     # chama o __init__ da mãe
        self.valor_minimo = valor_minimo

    def rodape(self):
        base = super().rodape()                       # reaproveita e acrescenta
        return f"{base}\n(filtro: valor >= {self.valor_minimo:,.2f})"


print(RelatorioComFiltro("Praças relevantes", dados, valor_minimo=100000).gerar())

In [ ]:
# isinstance e a hierarquia
r = RelatorioComFiltro("X", dados)
print("é RelatorioComFiltro?", isinstance(r, RelatorioComFiltro))
print("é RelatorioTexto?    ", isinstance(r, RelatorioTexto))
print("é Relatorio?         ", isinstance(r, Relatorio))
print("\nMRO (ordem de resolução de métodos):")
for c in RelatorioComFiltro.__mro__:
    print("   ", c.__name__)

## 6. Composição > Herança

> **"Prefira composição a herança."**
> — princípio de design que você vai ouvir a vida inteira

| | Herança | Composição |
|---|---------|------------|
| Relação | "**é um**" (Gato é um Animal) | "**tem um**" (Carro tem um Motor) |
| Acoplamento | Forte — mudança na mãe quebra as filhas | Fraco |
| Flexibilidade | Fixa na definição da classe | Trocável em tempo de execução |
| Combinação | Múltipla herança = complexidade | Junte quantas peças quiser |

**Por que herança dá problema:** ela cria uma dependência rígida. Quando você tem 4 dimensões de variação (formato × destino × filtro × idioma), a herança exige 4×3×2×2 = 48 subclasses. A composição resolve com 4+3+2+2 = 11 peças combináveis.

In [ ]:
# ❌ HERANÇA: explosão combinatória
class RelatorioTextoArquivo: pass
class RelatorioTextoEmail: pass
class RelatorioTextoS3: pass
class RelatorioCSVArquivo: pass
class RelatorioCSVEmail: pass
class RelatorioCSVS3: pass
class RelatorioJSONArquivo: pass
# ... e adicionar um formato novo exige 3 classes novas

print("Herança: 3 formatos × 3 destinos = 9 classes")
print("Adicionar 1 formato → +3 classes")
print("Adicionar 1 destino → +3 classes")

In [ ]:
# ✅ COMPOSIÇÃO: peças independentes que se combinam
class FormatadorTexto:
    extensao = "txt"
    def formatar(self, dados):
        return "\n".join(f"{d['cidade']:<20}{d['valor']:>15,.2f}" for d in dados)


class FormatadorCSV:
    extensao = "csv"
    def formatar(self, dados):
        linhas = ["cidade;valor"]
        linhas += [f"{d['cidade']};{d['valor']:.2f}" for d in dados]
        return "\n".join(linhas)


class FormatadorJSON:
    extensao = "json"
    def formatar(self, dados):
        import json
        return json.dumps(dados, ensure_ascii=False, indent=2)


class DestinoConsole:
    def entregar(self, conteudo, nome):
        print(f"── {nome} ──")
        print(conteudo[:200] + ("..." if len(conteudo) > 200 else ""))


class DestinoArquivo:
    def __init__(self, pasta="saida_aula"):
        from pathlib import Path
        self.pasta = Path(pasta)
        self.pasta.mkdir(exist_ok=True)

    def entregar(self, conteudo, nome):
        caminho = self.pasta / nome
        caminho.write_text(conteudo, encoding="utf-8")
        print(f"💾 gravado: {caminho} ({len(conteudo)} bytes)")


class DestinoMemoria:
    def __init__(self):
        self.arquivos = {}

    def entregar(self, conteudo, nome):
        self.arquivos[nome] = conteudo
        print(f"🧠 em memória: {nome}")


class Relatorio:
    """Não herda de nada. RECEBE as peças que precisa."""

    def __init__(self, titulo, formatador, destino):
        self.titulo = titulo
        self.formatador = formatador     # ← composição
        self.destino = destino           # ← composição

    def gerar(self, dados):
        conteudo = self.formatador.formatar(dados)
        nome = f"{self.titulo.lower().replace(' ', '_')}.{self.formatador.extensao}"
        self.destino.entregar(conteudo, nome)
        return conteudo


print("Composição: 3 formatadores + 3 destinos = 6 classes")
print("Combinações possíveis: 9")
print("Adicionar 1 formato → +1 classe (e ganha 3 combinações)\n")

# Qualquer combinação, montada em tempo de execução
Relatorio("Vendas", FormatadorTexto(), DestinoConsole()).gerar(dados)
print()
Relatorio("Vendas", FormatadorCSV(), DestinoArquivo()).gerar(dados)
Relatorio("Vendas", FormatadorJSON(), DestinoArquivo()).gerar(dados)

> 💡 **Duck typing.** Repare que `FormatadorTexto`, `FormatadorCSV` e `FormatadorJSON` **não herdam de nada em comum**. Eles funcionam porque todos têm o método `formatar` e o atributo `extensao`.
>
> *"Se anda como pato e grasna como pato, é um pato."* Python não exige declaração de interface — exige o comportamento.
>
> Isso torna trivial adicionar um formatador: escreva uma classe com `formatar` e `extensao`, e ela já funciona com todo o resto.

In [ ]:
# Injeção de dependência: trocar peças para testar
memoria = DestinoMemoria()
r = Relatorio("Teste", FormatadorCSV(), memoria)
r.gerar(dados)

print("\nConteúdo capturado (sem tocar no disco):")
print(list(memoria.arquivos.values())[0])

> 💭 **Este é o padrão que torna código testável.** No Módulo 12, você vai escrever testes que injetam um `DestinoMemoria` no lugar do `DestinoArquivo` — e verificam o conteúdo sem criar arquivos. Com herança rígida, isso seria impossível sem gambiarra.

## 7. Métodos dunder (*double underscore*)

Os métodos `__assim__` conectam sua classe aos **operadores e funções embutidas** do Python.

| Dunder | Ativado por | Para quê |
|--------|-------------|----------|
| `__init__` | `Classe(...)` | Inicializar |
| `__repr__` | `repr(x)`, console | Representação para **desenvolvedor** |
| `__str__` | `str(x)`, `print(x)` | Representação para **usuário** |
| `__eq__` | `x == y` | Igualdade |
| `__lt__`, `__gt__` | `<`, `>`, `sorted()` | Ordenação |
| `__hash__` | `hash(x)`, `set`, chave de `dict` | Hashabilidade |
| `__len__` | `len(x)` | Tamanho |
| `__getitem__` | `x[i]` | Indexação |
| `__contains__` | `y in x` | Pertencimento |
| `__iter__` | `for y in x` | Iteração |
| `__add__` | `x + y` | Operadores |
| `__call__` | `x(...)` | Objeto chamável |
| `__enter__`/`__exit__` | `with x:` | Context manager |
| `__bool__` | `if x:` | Veracidade |

In [ ]:
class Dinheiro:
    """Valor monetário com operações seguras.

    Guarda CENTAVOS como inteiro — evita o 0.1 + 0.2 != 0.3
    da aula 01_01.
    """

    def __init__(self, reais=0, centavos=0):
        self._centavos = int(round(reais * 100)) + centavos

    # ── Representação ──
    def __repr__(self):
        """Para o DESENVOLVEDOR. Idealmente reconstrói o objeto."""
        return f"Dinheiro({self._centavos / 100!r})"

    def __str__(self):
        """Para o USUÁRIO."""
        sinal = "-" if self._centavos < 0 else ""
        v = abs(self._centavos)
        return f"{sinal}R$ {v // 100:,}".replace(",", ".") + f",{v % 100:02d}"

    # ── Comparação ──
    def __eq__(self, outro):
        return isinstance(outro, Dinheiro) and self._centavos == outro._centavos

    def __lt__(self, outro):
        return self._centavos < outro._centavos

    def __hash__(self):
        # ⚠️ Definiu __eq__? DEFINA __hash__ também, ou o objeto
        #    deixa de ser hasheável e não serve como chave de dict.
        return hash(self._centavos)

    # ── Aritmética ──
    def __add__(self, outro):
        return Dinheiro(centavos=self._centavos + outro._centavos)

    def __sub__(self, outro):
        return Dinheiro(centavos=self._centavos - outro._centavos)

    def __mul__(self, fator):
        return Dinheiro(centavos=int(round(self._centavos * fator)))

    def __neg__(self):
        return Dinheiro(centavos=-self._centavos)

    def __bool__(self):
        return self._centavos != 0


a = Dinheiro(2599.90)
b = Dinheiro(89.90)

print(f"repr : {a!r}")
print(f"str  : {a}")
print(f"a + b: {a + b}")
print(f"a - b: {a - b}")
print(f"a * 3: {a * 3}")
print(f"-a   : {-a}")
print(f"a > b: {a > b}")
print(f"ordenado: {[str(x) for x in sorted([a, b, Dinheiro(1199)])]}")
print(f"em set : {len({Dinheiro(100), Dinheiro(100), Dinheiro(200)})} elementos distintos")
print(f"bool(0): {bool(Dinheiro(0))}")

In [ ]:
# ✅ A prova de que centavos-como-inteiro resolve o problema do float
soma_float = 0.1 + 0.2
soma_dinheiro = Dinheiro(0.1) + Dinheiro(0.2)

print(f"float   : {soma_float!r}  == 0.3? {soma_float == 0.3}")
print(f"Dinheiro: {soma_dinheiro}  == R$0,30? {soma_dinheiro == Dinheiro(0.3)}")

> 🔴 **`__repr__` vs `__str__` — a regra:**
>
> - `__repr__` é para **você**: deve ser inequívoco, idealmente código que reconstrói o objeto. É o que aparece no console e nos tracebacks.
> - `__str__` é para o **usuário final**: legível e bonito.
>
> **Se você só for definir um, defina `__repr__`** — o `__str__` cai de volta nele automaticamente. O contrário não acontece, e você acaba com `<__main__.Produto object at 0x7f8b...>` nos logs, que não ajuda ninguém.

In [ ]:
# Container: len, getitem, contains, iter
class Catalogo:
    def __init__(self, produtos=None):
        self._produtos = list(produtos or [])

    def __len__(self):
        return len(self._produtos)

    def __getitem__(self, indice):
        # Aceitar slice também é boa educação
        return self._produtos[indice]

    def __contains__(self, sku):
        return any(p.sku == sku for p in self._produtos)

    def __iter__(self):
        return iter(self._produtos)

    def __repr__(self):
        return f"Catalogo({len(self._produtos)} produtos)"

    def adicionar(self, produto):
        self._produtos.append(produto)
        return self         # permite encadeamento


catalogo = Catalogo([
    Produto("NB-01", "Notebook Dell", 2599.90, 2120.00, 14),
    Produto("PE-01", "Mouse Logitech", 89.90, 52.00, 240),
    Produto("MO-01", "Monitor LG", 1199.00, 920.00, 31),
])

print(f"repr        : {catalogo!r}")
print(f"len         : {len(catalogo)}")
print(f"catalogo[0] : {catalogo[0].nome}")
print(f"fatia [:2]  : {[p.sku for p in catalogo[:2]]}")
print(f"'NB-01' in  : {'NB-01' in catalogo}")
print(f"'XX-99' in  : {'XX-99' in catalogo}")
print("iteração    :", [p.sku for p in catalogo])
print(f"if catalogo : {bool(catalogo)}   ← usa __len__ na falta de __bool__")

In [ ]:
# __call__: o objeto vira função
class Conversor:
    """Objeto chamável com configuração própria."""

    def __init__(self, taxa, moeda_origem="BRL", moeda_destino="USD"):
        self.taxa = taxa
        self.origem = moeda_origem
        self.destino = moeda_destino
        self.conversoes = 0

    def __call__(self, valor):
        self.conversoes += 1
        return round(valor / self.taxa, 2)

    def __repr__(self):
        return f"Conversor({self.origem}→{self.destino}, {self.conversoes} usos)"


para_dolar = Conversor(5.42)

print(para_dolar(2599.90))
print(para_dolar(1199.00))
print(f"\n{para_dolar!r}")
print("É chamável?", callable(para_dolar))

> 💡 **`__call__` vs closure.** Ambos guardam configuração e produzem comportamento. A diferença: o objeto chamável pode ter **outros métodos e estado inspecionável** (`para_dolar.conversoes`). Uma closure é opaca.
>
> Use closure quando for só uma função configurada; use `__call__` quando houver estado que alguém precise consultar.

## 8. Classes abstratas e protocolos

Como garantir que quem implementa um "formatador" realmente forneça o método `formatar`?

| Abordagem | Como funciona | Verificação |
|-----------|---------------|-------------|
| **Duck typing** | Só use; se faltar, dá `AttributeError` | Em execução, tarde |
| **ABC** (`abc.ABC`) | Herança obrigatória, métodos abstratos | Ao instanciar |
| **Protocol** (`typing`) | Tipagem estrutural, sem herança | Estática (mypy) |

In [ ]:
from abc import ABC, abstractmethod


class Formatador(ABC):
    """Contrato: toda subclasse DEVE implementar formatar e extensao."""

    @property
    @abstractmethod
    def extensao(self) -> str:
        ...

    @abstractmethod
    def formatar(self, dados: list[dict]) -> str:
        ...

    # Métodos concretos são herdados normalmente
    def nome_arquivo(self, titulo: str) -> str:
        return f"{titulo.lower().replace(' ', '_')}.{self.extensao}"


class FormatadorHTML(Formatador):
    extensao = "html"

    def formatar(self, dados):
        linhas = ["<table>", "<tr><th>Cidade</th><th>Valor</th></tr>"]
        for d in dados:
            linhas.append(f"<tr><td>{d['cidade']}</td><td>{d['valor']:,.2f}</td></tr>")
        linhas.append("</table>")
        return "\n".join(linhas)


f = FormatadorHTML()
print(f.nome_arquivo("Vendas Julho"))
print(f.formatar(dados[:2]))

In [ ]:
# A ABC IMPEDE instanciar uma implementação incompleta
class FormatadorIncompleto(Formatador):
    extensao = "xml"
    # esqueceu de implementar formatar()


try:
    FormatadorIncompleto()
except TypeError as erro:
    print(f"❌ {erro}")

print("\n💡 O erro aparece na INSTANCIAÇÃO, não quando o método é chamado.")
print("   Falhar cedo é sempre melhor que falhar em produção.")

In [ ]:
# Protocol: contrato SEM herança (tipagem estrutural)
from typing import Protocol, runtime_checkable


@runtime_checkable
class Formatavel(Protocol):
    extensao: str
    def formatar(self, dados: list[dict]) -> str: ...


# Esta classe NÃO herda de nada — e ainda assim satisfaz o protocolo
class FormatadorYAML:
    extensao = "yaml"

    def formatar(self, dados):
        return "\n".join(f"- cidade: {d['cidade']}\n  valor: {d['valor']}" for d in dados)


print("FormatadorYAML satisfaz o protocolo?", isinstance(FormatadorYAML(), Formatavel))
print("FormatadorHTML satisfaz o protocolo?", isinstance(FormatadorHTML(), Formatavel))
print("Uma string satisfaz?                ", isinstance("texto", Formatavel))

> 🧭 **Qual escolher?**
>
> - **ABC** quando você controla as subclasses e quer compartilhar código concreto. Boa para hierarquias internas do seu projeto.
> - **Protocol** quando as implementações vêm de fora (ou de bibliotecas de terceiros) e você só quer descrever a forma esperada. É mais "pythônico" e não força herança.
> - **Nada** quando o projeto é pequeno. Duck typing puro é suficiente na maior parte do tempo.

## 9. 🔴 Quando **não** usar classes

A parte mais importante desta aula.

### ❌ Classe que só tem um método

Se a classe tem `__init__` e mais um método, **é uma função**.

In [ ]:
# ❌ Classe desnecessária
class CalculadoraDeFrete:
    def __init__(self, valor, uf):
        self.valor = valor
        self.uf = uf

    def calcular(self):
        if self.valor >= 500:
            return 0.0
        return 12.90 if self.uf == "SP" else 29.90


print(CalculadoraDeFrete(150, "SP").calcular())


# ✅ Função
def calcular_frete(valor, uf):
    if valor >= 500:
        return 0.0
    return 12.90 if uf == "SP" else 29.90


print(calcular_frete(150, "SP"))

### ❌ Classe usada só como namespace de constantes

Use um módulo, um `Enum` ou constantes de módulo.

In [ ]:
# ❌
class Config:
    HOST = "localhost"
    PORTA = 5432

# ✅ Para valores fixos: constantes de módulo
HOST = "localhost"
PORTA = 5432

# ✅ Para um conjunto fechado de opções: Enum
from enum import Enum

class Status(Enum):
    PAGO = "pago"
    PENDENTE = "pendente"
    CANCELADO = "cancelado"


print(Status.PAGO, "|", Status.PAGO.value)
print("Todos:", [s.value for s in Status])
print("Do valor:", Status("pago"))

### ❌ Classe que só carrega dados

Use `dataclass` (aula 04_04) ou `NamedTuple` — você escreve 3 linhas em vez de 30.

In [ ]:
# ❌ 25 linhas de código repetitivo
class PontoManual:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"PontoManual(x={self.x}, y={self.y})"
    def __eq__(self, outro):
        return isinstance(outro, PontoManual) and (self.x, self.y) == (outro.x, outro.y)
    def __hash__(self):
        return hash((self.x, self.y))


# ✅ 3 linhas, com repr, eq, hash e ordenação de graça
from dataclasses import dataclass

@dataclass(frozen=True, order=True)
class Ponto:
    x: float
    y: float


p1, p2 = Ponto(1, 2), Ponto(1, 2)
print(p1, "| iguais?", p1 == p2, "| hasheável?", len({p1, p2}) == 1)

### ✅ Quando a classe **é** a resposta

| Sinal | Exemplo no Atlas |
|-------|------------------|
| Estado + comportamento que andam juntos | `Conexao` que guarda o handle e sabe abrir/fechar |
| Várias implementações da mesma ideia | `FormatadorTexto`, `FormatadorCSV`, `FormatadorJSON` |
| Objeto com ciclo de vida | `Pedido` que muda de estado |
| Muitas funções recebendo os mesmos 3 argumentos | `Repositorio(conexao, config)` |
| Precisa de context manager, iteração ou operadores | `Dinheiro`, `Catalogo` |
| Quer injetar dependência para testar | `Relatorio(formatador, destino)` |

## 🔧 Prática guiada — Refatorando o Atlas para objetos

Vamos pegar o monstro de 800 linhas e transformá-lo em peças coesas.

In [ ]:
%%writefile atlas_oop.py
"""Atlas — versão orientada a objetos.

Demonstra o desenho que o projeto vai adotar no Módulo 04:
  - dataclasses para os MODELOS (dados)
  - classes com comportamento para os SERVIÇOS
  - composição e injeção de dependência para a montagem
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass, field


# ═══════════════════════════════════════════════════════════════
#  MODELOS — só dados, com poucas propriedades calculadas
# ═══════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class Produto:
    sku: str
    nome: str
    categoria: str
    preco: float
    custo: float

    @property
    def margem_unitaria(self) -> float:
        return round(self.preco - self.custo, 2)


@dataclass
class ItemVenda:
    produto: Produto
    quantidade: int
    preco_unitario: float

    @property
    def total(self) -> float:
        return round(self.quantidade * self.preco_unitario, 2)

    @property
    def margem(self) -> float:
        return round(self.quantidade * (self.preco_unitario - self.produto.custo), 2)


@dataclass
class Pedido:
    id: int
    cidade: str
    uf: str
    canal: str
    status: str
    itens: list[ItemVenda] = field(default_factory=list)

    @property
    def total(self) -> float:
        return round(sum(i.total for i in self.itens), 2)

    @property
    def margem(self) -> float:
        return round(sum(i.margem for i in self.itens), 2)

    @property
    def faturado(self) -> bool:
        return self.status == "pago"

    def adicionar(self, item: ItemVenda) -> Pedido:
        self.itens.append(item)
        return self          # encadeamento


# ═══════════════════════════════════════════════════════════════
#  SERVIÇOS — comportamento
# ═══════════════════════════════════════════════════════════════

class Agregador:
    """Agrega pedidos por qualquer dimensão.

    Repare: uma classe, não sete funções quase iguais. A dimensão
    vira PARÂMETRO, não uma cópia do código.
    """

    def __init__(self, pedidos: list[Pedido]):
        self.pedidos = pedidos

    @property
    def faturados(self) -> list[Pedido]:
        return [p for p in self.pedidos if p.faturado]

    def por(self, dimensao: str) -> dict[str, dict]:
        acumulado: dict[str, dict] = defaultdict(
            lambda: {"pedidos": 0, "itens": 0, "receita": 0.0, "margem": 0.0}
        )
        for pedido in self.faturados:
            chave = getattr(pedido, dimensao)
            a = acumulado[chave]
            a["pedidos"] += 1
            a["itens"] += sum(i.quantidade for i in pedido.itens)
            a["receita"] = round(a["receita"] + pedido.total, 2)
            a["margem"] = round(a["margem"] + pedido.margem, 2)
        return dict(acumulado)

    def por_produto(self) -> dict[str, dict]:
        acumulado: dict[str, dict] = defaultdict(
            lambda: {"unidades": 0, "receita": 0.0, "margem": 0.0}
        )
        for pedido in self.faturados:
            for item in pedido.itens:
                a = acumulado[item.produto.nome]
                a["unidades"] += item.quantidade
                a["receita"] = round(a["receita"] + item.total, 2)
                a["margem"] = round(a["margem"] + item.margem, 2)
        return dict(acumulado)

    def totais(self) -> dict:
        faturados = self.faturados
        receita = round(sum(p.total for p in faturados), 2)
        return {
            "pedidos_total": len(self.pedidos),
            "pedidos_faturados": len(faturados),
            "receita": receita,
            "margem": round(sum(p.margem for p in faturados), 2),
            "ticket_medio": round(receita / len(faturados), 2) if faturados else 0.0,
            "taxa_cancelamento": round(
                sum(1 for p in self.pedidos if p.status == "cancelado") / len(self.pedidos), 4
            ) if self.pedidos else 0.0,
        }


# ═══════════════════════════════════════════════════════════════
#  APRESENTAÇÃO — composição, não herança
# ═══════════════════════════════════════════════════════════════

class Formatador(ABC):
    @abstractmethod
    def formatar(self, titulo: str, tabela: dict[str, dict]) -> str: ...


class FormatadorTabela(Formatador):
    def __init__(self, ordenar_por: str = "receita"):
        self.ordenar_por = ordenar_por

    def formatar(self, titulo, tabela):
        if not tabela:
            return f"{titulo}\n(sem dados)"

        colunas = list(next(iter(tabela.values())).keys())
        largura = 20 + 14 * len(colunas)

        # Ordena pela coluna pedida; se ela não existir, pela última numérica.
        chave_ordem = self.ordenar_por if self.ordenar_por in colunas else colunas[-1]

        linhas = [titulo.center(largura), "─" * largura]
        linhas.append(f"{'chave':<20}" + "".join(f"{c:>14}" for c in colunas))
        linhas.append("─" * largura)

        for chave, valores in sorted(tabela.items(), key=lambda kv: -kv[1][chave_ordem]):
            linha = f"{str(chave):<20}"
            for c in colunas:
                v = valores[c]
                linha += f"{v:>14,.2f}" if isinstance(v, float) else f"{v:>14,}"
            linhas.append(linha)

        linhas.append("─" * largura)
        linhas.append(f"(ordenado por {chave_ordem})")
        return "\n".join(linhas)


class FormatadorMarkdown(Formatador):
    def formatar(self, titulo, tabela):
        if not tabela:
            return f"## {titulo}\n\n_sem dados_"
        colunas = list(next(iter(tabela.values())).keys())
        linhas = [f"## {titulo}", "",
                  "| chave | " + " | ".join(colunas) + " |",
                  "|---|" + "---:|" * len(colunas)]
        for chave, valores in tabela.items():
            celulas = [f"{valores[c]:,.2f}" if isinstance(valores[c], float)
                       else f"{valores[c]:,}" for c in colunas]
            linhas.append(f"| {chave} | " + " | ".join(celulas) + " |")
        return "\n".join(linhas)


class Relatorio:
    """Orquestra. Recebe suas dependências — não as constrói."""

    def __init__(self, agregador: Agregador, formatador: Formatador):
        self.agregador = agregador
        self.formatador = formatador

    def secao(self, titulo: str, dimensao: str) -> str:
        return self.formatador.formatar(titulo, self.agregador.por(dimensao))

    def completo(self) -> str:
        partes = [
            self.secao("FATURAMENTO POR CIDADE", "cidade"),
            self.secao("FATURAMENTO POR UF", "uf"),
            self.secao("FATURAMENTO POR CANAL", "canal"),
            self.formatador.formatar("RANKING DE PRODUTOS",
                                     self.agregador.por_produto()),
        ]
        return "\n\n".join(partes)

In [ ]:
import atlas_oop
import random

# Montando dados de teste
rnd = random.Random(42)

catalogo = [
    atlas_oop.Produto("NB-01", "Notebook Dell",  "Notebooks",   2599.90, 2120.00),
    atlas_oop.Produto("NB-02", "Notebook Acer",  "Notebooks",   3299.00, 2780.00),
    atlas_oop.Produto("MO-01", "Monitor LG",     "Monitores",   1199.00,  920.00),
    atlas_oop.Produto("PE-01", "Mouse Logitech", "Periféricos",   89.90,   52.00),
    atlas_oop.Produto("PE-02", "Teclado Redragon","Periféricos",  249.00,  150.00),
    atlas_oop.Produto("AR-01", "SSD 1TB",        "Armazenamento", 489.00,  360.00),
]

cidades = [("Campinas", "SP"), ("São Paulo", "SP"), ("Curitiba", "PR"),
           ("Salvador", "BA"), ("Recife", "PE")]

pedidos = []
for pid in range(1, 121):
    cidade, uf = rnd.choice(cidades)
    pedido = atlas_oop.Pedido(
        id=pid, cidade=cidade, uf=uf,
        canal=rnd.choices(["site", "app", "marketplace"], weights=[50, 30, 20])[0],
        status=rnd.choices(["pago", "pendente", "cancelado"], weights=[80, 12, 8])[0],
    )
    for produto in rnd.sample(catalogo, rnd.choices([1, 2, 3], weights=[55, 30, 15])[0]):
        pedido.adicionar(atlas_oop.ItemVenda(
            produto=produto,
            quantidade=rnd.choices([1, 2, 3, 5, 10], weights=[55, 22, 12, 7, 4])[0],
            preco_unitario=round(produto.preco * rnd.choice([1.0, 1.0, 0.95, 0.90]), 2),
        ))
    pedidos.append(pedido)

print(f"✅ {len(pedidos)} pedidos, {sum(len(p.itens) for p in pedidos)} itens\n")
print("Exemplo:", pedidos[0])
print(f"  total: R$ {pedidos[0].total:,.2f} | margem: R$ {pedidos[0].margem:,.2f}")

In [ ]:
agregador = atlas_oop.Agregador(pedidos)

print("TOTAIS")
print("─" * 44)
for chave, valor in agregador.totais().items():
    if isinstance(valor, float) and chave.startswith("taxa"):
        print(f"  {chave:<22}{valor:>18.1%}")
    elif isinstance(valor, float):
        print(f"  {chave:<22}{valor:>18,.2f}")
    else:
        print(f"  {chave:<22}{valor:>18,}")

In [ ]:
# A MESMA agregação, dois formatos — só troca a peça injetada
relatorio_texto = atlas_oop.Relatorio(agregador, atlas_oop.FormatadorTabela())
print(relatorio_texto.secao("FATURAMENTO POR CIDADE", "cidade"))

In [ ]:
relatorio_md = atlas_oop.Relatorio(agregador, atlas_oop.FormatadorMarkdown())
print(relatorio_md.secao("FATURAMENTO POR CANAL", "canal"))

In [ ]:
# Adicionando uma dimensão nova: ZERO código novo.
# 'uf' já é atributo do Pedido, então `por()` já sabe agregar por ele.
print(relatorio_texto.secao("FATURAMENTO POR UF", "uf"))

> 💭 **Volte à dor do início.** "14 funções passando os mesmos 3 argumentos."
>
> Agora: `Agregador` guarda os pedidos **uma vez**. Cada método usa `self.pedidos`. Adicionar uma dimensão de análise não cria função nova — vira **parâmetro** de `por()`. Trocar o formato de saída é trocar **um objeto injetado**.
>
> E repare no que **não** fizemos: nenhuma hierarquia de herança de 4 níveis, nenhuma `AbstractBaseFactoryManager`. Duas dataclasses, uma classe de serviço e duas de formatação.

## 📝 Exercícios

**E1.** Crie a classe `Cliente` com `nome`, `email`, `cidade`, `uf` e `segmento`. Adicione `__repr__`, `__eq__` (por e-mail) e `__hash__`. Prove que funciona em `set`.

**E2.** Adicione a `Cliente` uma `property` `email` com setter que normalize (`.strip().lower()`) e valide que contém `@`. Teste com entradas ruins.

**E3.** Crie `Estoque` com métodos `entrada(sku, qtd)`, `saida(sku, qtd)` e `saldo(sku)`. A saída deve levantar `EstoqueInsuficienteError` se não houver saldo. Use um `dict` interno.

**E4.** Crie `ContaCorrente` com `_saldo` privado, `depositar`, `sacar` (que recusa saldo negativo) e uma property `saldo` somente leitura. Adicione `__str__`.

**E5.** Escreva `Temperatura` que guarde celsius internamente e exponha `fahrenheit` e `kelvin` como properties **com setter** (atribuir em fahrenheit muda o celsius).

**E6.** Crie a hierarquia `Funcionario` → `Vendedor` (comissão sobre vendas) e `Gerente` (bônus fixo), com `calcular_salario()` sobrescrito. Use `super()` no `__init__`.

**E7.** Refaça o exercício 6 usando **composição**: `Funcionario` recebe uma estratégia de remuneração. Compare os dois desenhos em um comentário — qual você prefere e por quê?

**E8.** Implemente `Fracao` com `__add__`, `__sub__`, `__mul__`, `__eq__`, `__lt__`, `__repr__` e simplificação automática por MDC.

**E9.** Implemente `Vetor2D` com `__add__`, `__sub__`, `__mul__` (escalar), `__abs__` (magnitude), `__eq__` e `__repr__`.

**E10.** Crie `ListaOrdenada` que mantenha os elementos sempre ordenados na inserção, implementando `__len__`, `__getitem__`, `__contains__` (com busca binária!) e `__iter__`.

**E11.** Defina uma ABC `Notificador` com `enviar(mensagem)` abstrato, e três implementações: `NotificadorEmail`, `NotificadorSlack` e `NotificadorLog`. Escreva `alertar(notificadores, msg)` que use todas.

**E12.** Reescreva a classe abaixo eliminando o excesso de OOP:
```python
class ValidadorDeCPF:
    def __init__(self, cpf):
        self.cpf = cpf
    def validar(self):
        return len(self.cpf.replace(".", "").replace("-", "")) == 11
```

**E13.** Estenda `atlas_oop.py` com um `FormatadorJSON` e um `FormatadorCSV`. Prove que nenhuma outra classe precisou mudar.

**E14.** Adicione a `Agregador` um método `top(dimensao, n=5, por="receita")` que devolva os N maiores. Use `sorted` com `key`.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

## 📋 Cola de referência

```python
# ── Classe básica ──
class Nome:
    CONSTANTE = 42                    # atributo de CLASSE (só imutáveis!)

    def __init__(self, a):
        self.a = a                    # atributo de INSTÂNCIA

    def metodo(self):                 # recebe self
        return self.a

    @property
    def calculado(self):              # acessa sem parênteses
        return self.a * 2

    @calculado.setter
    def calculado(self, v):
        self.a = v / 2

    @classmethod
    def de_dict(cls, d):              # construtor alternativo
        return cls(**d)               # use cls, não o nome da classe

    @staticmethod
    def util(x):                      # sem self nem cls
        return x * 2

# ⚠️ Atributo de classe MUTÁVEL é compartilhado por todas as instâncias

# ── Herança ──
class Filha(Mae):
    def __init__(self, a, b):
        super().__init__(a)           # chama o __init__ da mãe
        self.b = b

    def metodo(self):
        return super().metodo() + "!"

Filha.__mro__                          # ordem de resolução

# ── Composição (prefira!) ──
class Servico:
    def __init__(self, dependencia):   # RECEBE, não constrói
        self.dep = dependencia

# ── Dunder essenciais ──
__init__  __repr__  __str__            # defina __repr__ no mínimo
__eq__  __lt__  __hash__               # definiu __eq__? defina __hash__
__len__  __getitem__  __contains__  __iter__
__add__  __sub__  __mul__  __neg__
__call__                               # objeto vira função
__enter__  __exit__                    # with
__bool__

# ── ABC ──
from abc import ABC, abstractmethod
class Base(ABC):
    @abstractmethod
    def obrigatorio(self): ...
# instanciar sem implementar → TypeError

# ── Protocol (tipagem estrutural, sem herança) ──
from typing import Protocol, runtime_checkable
@runtime_checkable
class Forma(Protocol):
    def metodo(self) -> str: ...

# ── Enum ──
from enum import Enum
class Status(Enum):
    PAGO = "pago"
Status.PAGO.value   Status("pago")   list(Status)

# ── 🔴 NÃO use classe quando ──
#   • só tem um método além de __init__      → função
#   • só carrega dados                       → dataclass / NamedTuple
#   • só agrupa constantes                   → módulo / Enum
```

## ✅ Checklist de saída

- [ ] Explico a diferença entre classe e instância, e o papel do `self`
- [ ] Sei que `__init__` inicializa, não constrói
- [ ] Diferencio atributo de classe de atributo de instância
- [ ] **Nunca uso mutável como atributo de classe**
- [ ] Uso `_` para sinalizar interno, sabendo que é convenção
- [ ] Uso `@property` para cálculo e validação, sem criar getters por reflexo
- [ ] Uso `@classmethod` para construtores alternativos, com `cls`
- [ ] Sei quando `@staticmethod` faz sentido (e que raramente faz)
- [ ] Uso `super()` para estender em vez de substituir
- [ ] **Prefiro composição a herança** e sei explicar por quê
- [ ] Entendo duck typing e injeção de dependência
- [ ] Defino `__repr__` em toda classe que criar
- [ ] Sei que definir `__eq__` exige definir `__hash__`
- [ ] Implemento `__len__`/`__getitem__`/`__iter__` em containers
- [ ] Sei escolher entre ABC, Protocol e duck typing puro
- [ ] **Reconheço quando uma classe deveria ser uma função**

---

### ➡️ Próxima aula

**`04_04_Tipagem_e_Estruturas.ipynb`** — Dataclasses, type hints, mypy e Pydantic. Onde o Python ganha uma rede de segurança sem perder a flexibilidade.